# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravindrathalari06/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-06 Section 1: inspect key signal distributions

import pandas as pd

url = "https://raw.githubusercontent.com/Ravindrathalari06/flyrank-internship-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

key_signals = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate"
]

print("Key signal distributions:")

distribution_summary = df[key_signals].describe().T[
    ["count", "mean", "50%", "min", "max"]
]

distribution_summary["missing"] = df[key_signals].isna().sum()

display(distribution_summary)

Key signal distributions:


,count,mean,50%,min,max,missing
search_volume,27532.0,158.882391,10.00,0.0,74000.0,2468
impressions_90d,30000.0,5200.366300,731.00,1.0,517715.0,0
clicks_90d,30000.0,16.097333,1.00,0.0,4178.0,0
sessions_90d,30000.0,37.066633,7.00,1.0,4345.0,0
content_age_days,30000.0,256.167800,236.00,90.0,564.0,0
days_since_last_update,30000.0,46.098300,20.00,1.0,373.0,0
ctr,30000.0,0.510733,0.07,0.0,100.0,0
avg_position,30000.0,16.342380,10.80,0.0,245.0,0
engagement_rate,30000.0,2.534520,0.00,0.0,100.0,0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [11]:
# ML-06 Section 2: test three safe signals

tests = [
    "days_since_last_update",
    "impressions_last_30d",
    "clicks_last_30d"
]

for signal in tests:
    print("\n" + "=" * 60)
    print("Signal:", signal)

    summary = (
        df.groupby("trend_direction")[signal]
        .agg(["count", "median", "mean"])
    )

    display(summary)

    medians = summary["median"].dropna()

    if len(medians) >= 2 and medians.nunique() > 1:
        verdict = "MIXED"
        reason = "Observed group differences exist, but the trend labels are categorical and do not define an ordered scale."
    else:
        verdict = "FALSE"
        reason = "No useful group difference observed."

    print("Verdict:", verdict)
    print("Reason:", reason)


Signal: days_since_last_update


,count,median,mean
trend_direction,,,
down,16262,20.0,49.245788
flat,1152,20.0,37.453993
new,2236,20.0,22.337209
stable,5962,22.0,50.061892
up,4388,22.0,43.425706


Verdict: MIXED
Reason: Observed group differences exist, but the trend labels are categorical and do not define an ordered scale.

Signal: impressions_last_30d


,count,median,mean
trend_direction,,,
down,16262,128.0,941.556635
flat,1152,0.0,0.000000
new,2236,2.0,160.454383
stable,5962,543.0,2962.575310
up,4388,229.0,2173.773473


Verdict: MIXED
Reason: Observed group differences exist, but the trend labels are categorical and do not define an ordered scale.

Signal: clicks_last_30d


,count,median,mean
trend_direction,,,
down,16262,0.0,3.351740
flat,1152,0.0,0.000000
new,2236,0.0,0.337209
stable,5962,1.0,11.079336
up,4388,0.0,6.085005


Verdict: MIXED
Reason: Observed group differences exist, but the trend labels are categorical and do not define an ordered scale.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-06 Section 3: flag-linked test

signal = "days_since_last_update"

print("Flag-linked signal:", signal)

flag_summary = (
    df.groupby("trend_direction")[signal]
    .agg(["count", "median", "mean", "min", "max"])
    .sort_values("median")
)

display(flag_summary)

# Compare pages with shorter vs longer time since update
median_update_gap = df[signal].median()

df["update_gap_group"] = df[signal].apply(
    lambda x: "Longer gap" if x > median_update_gap else "Shorter/equal gap"
)

flag_test = (
    df.groupby("update_gap_group")["trend_pct"]
    .agg(["count", "median", "mean"])
)

print("\nTrend percentage by update-gap group:")
display(flag_test)

print(
    "\nVerdict: MIXED — the observed relationship should be treated "
    "as directional evidence, not causal proof."
)

Flag-linked signal: days_since_last_update


,count,median,mean,min,max
trend_direction,,,,,
down,16262,20.0,49.245788,1,373
flat,1152,20.0,37.453993,1,373
new,2236,20.0,22.337209,1,372
stable,5962,22.0,50.061892,1,305
up,4388,22.0,43.425706,1,313



Trend percentage by update-gap group:


,count,median,mean
update_gap_group,,,
Longer gap,13620,-27.1,-8.488759
Shorter/equal gap,12992,-41.4,-0.904195



Verdict: MIXED — the observed relationship should be treated as directional evidence, not causal proof.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-06 Section 4: practical summary check

print("Practical takeaway:")
print("Use multiple observed signals for content prioritization.")
print("Treat signal relationships as directional, not causal.")
print("Keep final investigation decisions with the content team.")

Practical takeaway:
Use multiple observed signals for content prioritization.
Treat signal relationships as directional, not causal.
Keep final investigation decisions with the content team.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.